# Notebook 09 — LightGBM

Algoritmo: LightGBM. Escalado: no. n_iter = 50, CV=5, scoring='roc_auc'.

Observaciones: Variante eficiente de gradient boosting.

## 1. Setup

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ))

from src.preprocesamiento import preparar, RANDOM_STATE
from src.entrenamiento  import entrenar_y_evaluar

DATA = RAIZ / 'data'

from lightgbm import LGBMClassifier
from scipy.stats import loguniform, uniform

## 2. Cargar y preparar datos

In [2]:
datos = preparar(DATA, aplicar_escalado=False)
print(f'Train: {datos.n_train} | Test: {datos.n_test} | Features: {len(datos.cols_features)}')
print(f'scale_pos_weight atraso: {datos.scale_pos_weight["tuvo_atraso"]:.4f}')


# LGBM no acepta caracteres especiales JSON en nombres de cols. Sanitizar.
import re
def _clean(name): return re.sub(r'[^A-Za-z0-9_]+', '_', str(name))
datos.X_train.columns = [_clean(c) for c in datos.X_train.columns]
datos.X_test.columns  = [_clean(c) for c in datos.X_test.columns]


Train: 38664 | Test: 9667 | Features: 152
scale_pos_weight atraso: 0.6534


## 3. Clasificador + grid

In [3]:
clf = LGBMClassifier(objective='binary', n_jobs=-1, random_state=RANDOM_STATE,
    scale_pos_weight=datos.scale_pos_weight['tuvo_atraso'], verbose=-1)

param_distributions = {
        'n_estimators': [300, 500, 800],
        'learning_rate': loguniform(0.01, 0.3),
        'num_leaves': [31, 63, 127],
        'max_depth': [-1, 8, 10, 12],
        'min_child_samples': [10, 20, 50],
        'subsample': uniform(0.6, 0.4),
        'colsample_bytree': uniform(0.6, 0.4),
    }

## 4. RandomizedSearchCV (50 iters × CV 5)

In [4]:
resultado = entrenar_y_evaluar(
    nombre='lgbm',
    estimador=clf,
    param_distributions=param_distributions,
    datos=datos,
    data_dir=DATA,
    n_iter=50,
    target='tuvo_atraso',
)
resultado


=== LGBM ===
CV AUC:          0.9094
Test AUC:        0.9091
Test F1:         0.8503
Test Precision:  0.873
Test Recall:     0.8288
Test Accuracy:   0.8235
Tiempo search:   379.78 s
Best params:     {'colsample_bytree': 0.8159364365206693, 'learning_rate': 0.019950144748611585, 'max_depth': -1, 'min_child_samples': 50, 'n_estimators': 800, 'num_leaves': 127, 'subsample': 0.6644885149016018}


{'nombre': 'lgbm',
 'target': 'tuvo_atraso',
 'n_iter': 50,
 'cv_folds': 5,
 'tiempo_search_seg': 379.78,
 'best_params': {'colsample_bytree': 0.8159364365206693,
  'learning_rate': 0.019950144748611585,
  'max_depth': -1,
  'min_child_samples': 50,
  'n_estimators': 800,
  'num_leaves': 127,
  'subsample': 0.6644885149016018},
 'best_cv_auc': 0.9094,
 'AUC': 0.9091,
 'F1': 0.8503,
 'Precision': 0.873,
 'Recall': 0.8288,
 'Accuracy': 0.8235}